# NYC Census Block Group Adjacency Matrix Generation

This notebook generates and visualizes the spatial adjacency network for Census Block Groups (CBGs) in NYC.
It uses a geometric buffer approach to define adjacency relationships.


In [ ]:
import sys
sys.path.insert(0, '../../for_paper/adjacency')
sys.path.insert(0, '../../..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from tract_weights import GeometryWeightsGenerator
from geometry_config import GeometryType, get_geometry_paths
from IPython.display import display, HTML

%matplotlib inline


## Generate Weights

First, let's compute the custom geometric weights matrix for Census Block Groups.
We use a 300ft buffer distance (smaller than the 500ft used for Census Tracts due to smaller geometry sizes).


In [ ]:
# Get paths for CBG geometry
paths = get_geometry_paths(GeometryType.CBG)
print(f"CBG GeoJSON: {paths.geojson_path}")
print(f"Adjacency output dir: {paths.adjacency_dir}")


In [ ]:
# Initialize weights generator for Census Block Groups
generator = GeometryWeightsGenerator(GeometryType.CBG)

# Compute custom geometric weights with 300ft buffer
# (smaller buffer than CT due to smaller geometry sizes)
custom_geometric_weights = generator.compute_custom_geometric_weights(buffer_dist=300, debug=True)


In [ ]:
# Inspect the weights results
custom_geometric_weights


In [ ]:
# Print summary statistics
cg = custom_geometric_weights['custom_geometric']
print(f"=== CBG Adjacency Network Statistics ===")
print(f"Total CBGs: {cg.weights_matrix.shape[0]}")
print(f"Total connections: {cg.n_connections}")
print(f"Average connections per CBG: {cg.avg_connections:.2f}")
print(f"Isolated CBGs: {len(cg.isolated_geometries)}")
if cg.isolated_geometries:
    print(f"Isolated CBG indices: {cg.isolated_geometries[:10]}..." if len(cg.isolated_geometries) > 10 else f"Isolated CBG indices: {cg.isolated_geometries}")


In [ ]:
# Export weights to adjacency list files
output_dir = paths.adjacency_dir
os.makedirs(output_dir, exist_ok=True)
node1_path, node2_path = generator.export_adjacency_lists("custom_geometric", output_dir=output_dir)
print(f"Exported adjacency lists to:")
print(f"  {node1_path}")
print(f"  {node2_path}")


## Visualize Adjacency Network


In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

# Set up plotting style
plt.rcParams['figure.dpi'] = 100

def create_academic_map(gdf, adj_matrix, method_name, isolated_nodes=None, sample_frac=0.3):
    """
    Create a map visualization of the adjacency network.
    
    Parameters
    ----------
    gdf : GeoDataFrame
        Census geometry
    adj_matrix : ndarray
        Adjacency matrix
    method_name : str
        Name for the plot title
    isolated_nodes : list, optional
        Indices of isolated nodes to highlight
    sample_frac : float, optional
        Fraction of edges to plot (for performance with large networks)
    """
    # Project to Web Mercator for visualization
    gdf_proj = gdf.to_crs('EPSG:3857')
    
    fig, ax = plt.subplots(figsize=(12, 12))
    
    # Plot the census boundaries
    if 'BoroName' in gdf_proj.columns:
        gdf_proj.plot(ax=ax, column='BoroName', categorical=True, 
                     alpha=0.4, legend=True, legend_kwds={'loc': 'upper left'})
        leg = ax.get_legend()
        leg.set_title('Borough')
    else:
        gdf_proj.plot(ax=ax, alpha=0.4, color='lightblue', edgecolor='white', linewidth=0.1)
    
    # Sample edges for performance (CBG has many more edges than CT)
    edges_to_plot = []
    for i in range(adj_matrix.shape[0]):
        for j in range(i+1, adj_matrix.shape[1]):
            if adj_matrix[i,j] == 1:
                edges_to_plot.append((i, j))
    
    # Sample edges if there are too many
    n_edges = len(edges_to_plot)
    if n_edges > 5000:
        np.random.seed(42)
        sample_size = int(n_edges * sample_frac)
        edges_to_plot = [edges_to_plot[i] for i in np.random.choice(n_edges, sample_size, replace=False)]
        print(f"Plotting {len(edges_to_plot)} of {n_edges} edges ({sample_frac*100:.0f}% sample)")
    
    # Plot adjacency lines
    for i, j in edges_to_plot:
        cent1 = gdf_proj.iloc[i].geometry.centroid
        cent2 = gdf_proj.iloc[j].geometry.centroid
        ax.plot([cent1.x, cent2.x], [cent1.y, cent2.y], 
               color='blue', alpha=0.3, linewidth=0.3)
    
    # Highlight isolated nodes
    if isolated_nodes:
        for idx in isolated_nodes:
            centroid = gdf_proj.iloc[idx].geometry.centroid
            ax.plot(centroid.x, centroid.y, 'ro', markersize=3, alpha=0.7)
        print(f"Highlighted {len(isolated_nodes)} isolated nodes in red")
    
    ax.set_title(f'{method_name}\n{len(edges_to_plot)} edges shown', fontsize=14)
    ax.set_axis_off()
    
    return fig, ax


In [ ]:
# Load CBG geometry for visualization
gdf = gpd.read_file(str(paths.geojson_path))
gdf = gdf.to_crs(epsg=2263)

print(f"Loaded {len(gdf)} Census Block Groups")


In [ ]:
# Create visualization
adj_matrix = custom_geometric_weights['custom_geometric'].weights_matrix
isolated_nodes = custom_geometric_weights['custom_geometric'].isolated_geometries

fig, ax = create_academic_map(
    gdf, 
    adj_matrix, 
    'NYC Census Block Group Adjacency Network (300ft buffer)',
    isolated_nodes=isolated_nodes,
    sample_frac=0.2  # Plot 20% of edges for readability
)

plt.tight_layout()
plt.show()


## Compare with Census Tract Network


In [ ]:
# Load CT adjacency for comparison
ct_paths = get_geometry_paths(GeometryType.CT)
ct_node1 = ct_paths.adjacency_node1_path('custom_geometric')

if ct_node1.exists():
    with open(ct_node1) as f:
        ct_n_edges = len(f.readlines())
    
    cbg_n_edges = cg.n_connections
    
    print("=== Network Comparison ===")
    print(f"Census Tracts:")
    print(f"  - 2,325 areas")
    print(f"  - {ct_n_edges} edges")
    print(f"  - ~{ct_n_edges/2325:.1f} avg connections")
    print()
    print(f"Census Block Groups:")
    print(f"  - {cg.weights_matrix.shape[0]} areas")
    print(f"  - {cbg_n_edges} edges")
    print(f"  - ~{cg.avg_connections:.1f} avg connections")
    print()
    print(f"Ratio: CBG has {cbg_n_edges/ct_n_edges:.1f}x more edges")
else:
    print("CT adjacency not found for comparison")
